### Day 2 - Part 4 과제: 나만의 CNN 분류기 설계 및 성능 개선

`목표`: 이 과제는 튜토리얼에서 배운 CNN의 기본 구성 요소를 활용하여, CIFAR-10 데이터셋에 대한 분류 모델의 아키텍처를 직접 수정하고 실험함으로써 성능을 개선하는 것을 목표로 합니다.

`지시사항`:

1.  아래에 제공된 기본 코드를 실행하여 데이터셋을 준비하고 기준 모델의 성능을 확인합니다.
2.  `문제 1`부터 `문제 3`까지, 각 문제의 지시에 따라 `TODO` 또는 `# 코드를 입력하세요` 부분을 채워 새로운 CNN 모델을 설계하고 학습시킨 후 성능을 비교 분석합니다.
3.  `도전 과제`에서는 여러분이 배운 지식을 총동원하여 자유롭게 모델을 설계하고 최고의 분류 정확도에 도전해 보세요.

-----

In [ ]:
# 필요한 라이브러리 임포트
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchsummary import summary

# --- Part 1: 기본 코드 및 데이터 준비 (수정 불필요) ---

# Device 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 데이터 전처리 정의
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# CIFAR-10 데이터셋 로드
path = "../datasets/dl/cifar10/" # 로컬/도커 환경에 따라 알맞게 경로 변경 할것.
train_dataset = torchvision.datasets.CIFAR10(root=path, train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.CIFAR10(root=path, train=False, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

# 클래스 이름
classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

# 학습 및 평가 함수
def train_and_evaluate(model, train_loader, test_loader, num_epochs=15):
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    for epoch in range(num_epochs):
        model.train()
        for i, (inputs, labels) in enumerate(train_loader):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        print(f'Epoch {epoch + 1}/{num_epochs} completed.')

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f'Final Test Accuracy: {accuracy:.2f} %')
    return accuracy

# --- Part 2: 기준 CNN 모델 (튜토리얼의 SimpleCNN) ---

class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv_block1 = nn.Sequential(
            nn.Conv2d(3, 16, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2, 2)
        )
        self.conv_block2 = nn.Sequential(
            nn.Conv2d(16, 32, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2, 2)
        )
        self.fc_block = nn.Sequential(
            nn.Linear(32 * 8 * 8, 120), nn.ReLU(),
            nn.Linear(120, 84), nn.ReLU(),
            nn.Linear(84, 10)
        )

    def forward(self, x):
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        x = x.view(x.size(0), -1)
        x = self.fc_block(x)
        return x

print("--- Baseline Model (SimpleCNN) ---")
baseline_model = SimpleCNN()
summary(baseline_model, (3, 32, 32), device='cpu')
# baseline_accuracy = train_and_evaluate(baseline_model, train_loader, test_loader) # 시간 관계상 주석 처리, 필요시 실행

# --- Part 3: 과제 ---

### 문제 1: 모델 깊게 만들기 (Deeper CNN)
# 지시: SimpleCNN에 [Conv-ReLU-Pool] 블록을 하나 더 추가하여 3개의 합성곱 블록을 가진 DeeperCNN을 만드세요.
# 힌트: 세 번째 블록의 out_channels를 64로 설정해보고, 풀링 후 특징 맵 크기(4x4)를 고려하여 fc_block의 입력 크기를 다시 계산해야 합니다.

class DeeperCNN(nn.Module):
    def __init__(self):
        super(DeeperCNN, self).__init__()
        # 3개의 합성곱 블록과 1개의 완전연결 블록을 정의
        self.conv_block1 = nn.Sequential(
            nn.Conv2d(3, 16, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2, 2)
        )
        self.conv_block2 = nn.Sequential(
            nn.Conv2d(16, 32, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2, 2)
        )
        self.conv_block3 = nn.Sequential(
            nn.Conv2d(32, 64, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2, 2)
        )
        self.fc_block = nn.Sequential(
            nn.Linear(64 * 4 * 4, 120), nn.ReLU(),
            nn.Linear(120, 84), nn.ReLU(),
            nn.Linear(84, 10)
        )

    def forward(self, x):
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        x = self.conv_block3(x)
        x = x.view(x.size(0), -1)
        x = self.fc_block(x)
        return x

print("\n--- Assignment 1: DeeperCNN ---")
deeper_model = DeeperCNN()
summary(deeper_model, (3, 32, 32), device='cpu')
# deeper_accuracy = train_and_evaluate(deeper_model, train_loader, test_loader)


### 문제 2: 모델 넓게 만들기 (Wider CNN)
# 지시: SimpleCNN을 기반으로 각 합성곱 블록의 채널 수를 2배로 늘린 (16->32, 32->64) WiderCNN을 만드세요.
# 힌트: 채널 수만 변경되므로 fc_block의 입력 크기도 그에 맞게 수정되어야 합니다.

class WiderCNN(nn.Module):
    def __init__(self):
        super(WiderCNN, self).__init__()
        # 더 많은 채널을 가진 2개의 합성곱 블록과 1개의 완전연결 블록을 정의
        self.conv_block1 = nn.Sequential(
            nn.Conv2d(3, 32, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2, 2)
        )
        self.conv_block2 = nn.Sequential(
            nn.Conv2d(32, 64, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2, 2)
        )
        self.fc_block = nn.Sequential(
            nn.Linear(64 * 8 * 8, 120), nn.ReLU(),
            nn.Linear(120, 84), nn.ReLU(),
            nn.Linear(84, 10)
        )

    def forward(self, x):
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        x = x.view(x.size(0), -1)
        x = self.fc_block(x)
        return x

print("\n--- Assignment 2: WiderCNN ---")
wider_model = WiderCNN()
summary(wider_model, (3, 32, 32), device='cpu')
# wider_accuracy = train_and_evaluate(wider_model, train_loader, test_loader)


### 문제 3: 규제 기법 추가하기 (CNN with Dropout)
# 지시: DeeperCNN 또는 WiderCNN 모델에 nn.Dropout(p=0.5)을 완전연결 블록에 추가하여 과적합을 방지해 보세요.
# 힌트: 드롭아웃은 보통 nn.Linear와 nn.ReLU 사이에 위치시킵니다.

class CnnWithDropout(nn.Module):
    def __init__(self):
        super(CnnWithDropout, self).__init__()
        # DeeperCNN 구조를 기반으로, fc_block 내에 Dropout 레이어를 추가
        self.conv_block1 = nn.Sequential(
            nn.Conv2d(3, 16, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2, 2)
        )
        self.conv_block2 = nn.Sequential(
            nn.Conv2d(16, 32, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2, 2)
        )
        self.conv_block3 = nn.Sequential(
            nn.Conv2d(32, 64, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2, 2)
        )
        self.fc_block = nn.Sequential(
            nn.Linear(64 * 4 * 4, 120), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(120, 84), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(84, 10)
        )

    def forward(self, x):
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        x = self.conv_block3(x)
        x = x.view(x.size(0), -1)
        x = self.fc_block(x)
        return x

print("\n--- Assignment 3: CNN with Dropout ---")
regulated_model = CnnWithDropout()
summary(regulated_model, (3, 32, 32), device='cpu')
# regulated_accuracy = train_and_evaluate(regulated_model, train_loader, test_loader)


### 도전 과제: 자유롭게 최고의 모델 설계하기
# 지시: 위에서 배운 개념(깊이, 너비, 드롭아웃)과 추가적인 아이디어(예: 커널 크기 변경, 패딩/스트라이드 조절, 배치 정규화(nn.BatchNorm2d) 추가 등)를 자유롭게 조합하여 최고의 성능을 내는 모델을 만들어 보세요.

class MyBestCNN(nn.Module):
    def __init__(self):
        super(MyBestCNN, self).__init__()
        # 배치 정규화, 드롭아웃, 다양한 커널 크기를 조합한 최적화된 모델
        self.conv_block1 = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), 
            nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), 
            nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2, 2), nn.Dropout2d(0.25)
        )
        self.conv_block2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1), 
            nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), 
            nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2, 2), nn.Dropout2d(0.25)
        )
        self.conv_block3 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1), 
            nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), 
            nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2, 2), nn.Dropout2d(0.25)
        )
        self.fc_block = nn.Sequential(
            nn.Linear(128 * 4 * 4, 512), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        x = self.conv_block3(x)
        x = x.view(x.size(0), -1)
        x = self.fc_block(x)
        return x

print("\n--- Challenge: My Best CNN ---")
my_model = MyBestCNN()
summary(my_model, (3, 32, 32), device='cpu')
# my_best_accuracy = train_and_evaluate(my_model, train_loader, test_loader)